In [1]:
# modules
import os
import time

# local
from load_documents import ingest

from dotenv import load_dotenv

from constants import CHROMA_SETTINGS
from langchain.vectorstores import Chroma

from langchain.llms import GPT4All
from langchain.chains import RetrievalQA
from langchain.embeddings import HuggingFaceEmbeddings

from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler

In [2]:
# load environment
load_dotenv()

# general settings
verbose = False
embeddings_model_name = os.environ.get("EMBEDDINGS_MODEL_NAME")
persist_directory = os.environ.get('PERSIST_DIRECTORY')

# model settings
model_type = os.environ.get('MODEL_TYPE')
model_path = os.environ.get('MODEL_PATH')

# set maximum token limit for the LLM model
max_tokens = os.environ.get('MODEL_MAX_TOKENS')

# set number of tokens in the prompt that are passed into the model at a time
n_batch = int(os.environ.get('MODEL_N_BATCH', 8))

# set amount of sources (chunks) that will be used to answer a question
target_sources = int(os.environ.get('TARGET_SOURCES', 4))

# Chat with an LLM about uploaded source documents
Interact locally with documents using Generative Pre-trained Transformers.

### Setup

In [3]:
# load documents
ingest()

Creating new vectorstore
Loading documents from documents


Loading new documents: 100%|██████████████████████| 1/1 [00:02<00:00,  2.34s/it]


Loaded 9 new documents from documents
Split into 51 chunks of text (max. 500 tokens each)
Creating embeddings. May take some minutes...
Ingestion complete! You can now query your documents.


In [4]:
# create word embeddings
embeddings = HuggingFaceEmbeddings(model_name=embeddings_model_name)

In [5]:
# create a vectorized database to store contents
database = Chroma(persist_directory=persist_directory, embedding_function=embeddings, client_settings=CHROMA_SETTINGS)

# set up a db retriever to look for an amount of sources from embeddings that will be used to answer a question
retriever = database.as_retriever(search_kwargs={"k": target_sources})

In [6]:
# enable verbose mode for LLM callback to standard out
callbacks = list() #[StreamingStdOutCallbackHandler()]

### LLM

In [7]:
llm = GPT4All(model=model_path, max_tokens=max_tokens, backend='gptj', n_batch=n_batch, callbacks=callbacks, verbose=False)

Found model file at  models/ggml-gpt4all-j-v1.3-groovy.bin


In [8]:
# create a question-answer retrieval chain with MapReduce chain type
qa = RetrievalQA.from_chain_type(llm=llm, chain_type="stuff", retriever=retriever, return_source_documents=verbose)

### DocGPT

In [9]:
def run():
    """ Interact with documents loaded in the chroma database. """

    while True:
        print("> Query:")
        query = input()
    
        if query.lower() == "exit":
            break
            
        if query.strip() == "":
            print("\nPlease enter a valid query...\n\n")
            continue
    
        start = time.time()
    
        # get answer using RetrievalQA chain
        output = qa(query)
        answer = output.get("result")
    
        end = time.time()
    
        # display answer
        print("\n> Answer:")
        print(answer)
        print(f"\nETA: {round(end - start, 2)} (s)\n")

In [ ]:
run()

> Query:


 What is bitcoin?



> Answer:
 Bitcoin is an electronic cash system that allows for peer-to-peer transactions without the need of intermediaries such as banks. It uses digital signatures to ensure secure communication between parties involved in a transaction. The main benefit of this type of payment method is its ability to send payments directly from one party to another, rather than going through a financial institution like traditional banking systems do.

ETA: 71.1 (s)

> Query:


 Who created bitcoin?



> Answer:
 Satoshi Nakamoto

ETA: 57.78 (s)

> Query:


 What is bitcoin incentive?



> Answer:
 The Bitcoin Incentive refers to an economic mechanism in which transactions are rewarded for their inclusion in blocks and validated by nodes within the network. This incentivizes miners to continue mining new bitcoins as well as validating transactions on behalf of users, thereby maintaining a secure and stable digital currency system that is free from central control or manipulation.

ETA: 61.96 (s)

> Query:


 What is proof of work?



> Answer:
 Proof of Work (PoW) is a system used in blockchain technology that requires nodes on a network to solve complex mathematical problems known as "proofs" before they are allowed to add new transactions or make changes to existing ones, thus ensuring the integrity and security of the ledger. The proof-of-work involves scanning for a value that when hashed with SHA256 (or another hash function), the resulting hash begins with a number of zero bits. This process is repeated multiple times until it finds one such value in each block, which can be verified by executing a single hash and verifying all subsequent blocks on top of it to ensure consistency across the network. The proof-of-work system also solves the problem of determining representation in majority decision making, as nodes accept or reject transactions based on their work effort invested into creating new blocks that are part of the longest chain.

ETA: 96.1 (s)

> Query:
